In [2]:
import numpy as np
import pandas as pd
import sklearn as sk

In [20]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print(df['label'].unique())

df['label'] = df['label'].map({'ham': 0, 'spam': 1})
print(df['label'].unique())

X = df['message']
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
cv = CountVectorizer()
X_train_vec = cv.fit_transform(X_train)
X_test_vec  = cv.transform(X_test)

['ham' 'spam']
[0 1]


In [21]:
df.head(15)

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [6]:
df


,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will ü b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [23]:
class NaiveBayes:
    def fit(self, X, y):
        X = np.asarray(X.todense())
        total_messages = len(y)
        self.p_spam = np.sum(y == 1) / total_messages # compute probabilty of spam and normal
        self.p_ham = np.sum(y == 0) / total_messages

        X_spam = X[y == 1] #matrix of spam and normal messages encoded
        X_ham = X[y == 0]

        word_count_spam = X_spam.sum(axis=0) # sum of the word's occurence in many messages
        word_count_ham = X_ham.sum(axis=0)

        total_word_count_spam = word_count_spam.sum() # total number of words in many messages
        total_word_count_ham = word_count_ham.sum()

        vocab_size = X.shape[1]
        self.likelihood_spam = (word_count_spam + 1) / (total_word_count_spam + vocab_size) # P(word|spam) and laplace smoothening
        self.likelihood_ham = (word_count_ham + 1) / (total_word_count_ham + vocab_size) # P(word|ham)

    def predict(self, X):
        X = np.asarray(X.todense())
        predictions = []
        for i in X:
            log_score_spam = np.log(self.p_spam) + np.sum(i * np.log(self.likelihood_spam)) # Usage of log to avoid loss of very small probabilistic value
            log_score_ham = np.log(self.p_ham) + np.sum(i * np.log(self.likelihood_ham))
            if log_score_spam > log_score_ham:
                predictions.append(1)
            else:
                predictions.append(0)
        return predictions



In [24]:

nb = NaiveBayes()
nb.fit(X_train_vec, y_train)
predictions = nb.predict(X_test_vec)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

print("Accuracy:",  accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:",    recall_score(y_test, predictions))
print("F1 Score:",  f1_score(y_test, predictions))
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))

Accuracy: 0.9919282511210762
Precision: 1.0
Recall: 0.9395973154362416
F1 Score: 0.9688581314878892
Confusion Matrix:
 [[966   0]
 [  9 140]]


In [25]:
from sklearn.naive_bayes import MultinomialNB

sk_nb = MultinomialNB()
sk_nb.fit(X_train_vec, y_train)
sk_predictions = sk_nb.predict(X_test_vec)

print("Accuracy:",  accuracy_score(y_test, sk_predictions))
print("Precision:", precision_score(y_test, sk_predictions))
print("Recall:",    recall_score(y_test, sk_predictions))
print("F1 Score:",  f1_score(y_test, sk_predictions))

Accuracy: 0.9919282511210762
Precision: 1.0
Recall: 0.9395973154362416
F1 Score: 0.9688581314878892
